In [5]:
%reload_ext autoreload
%autoreload 2

import numpy as np
from matplotlib import pyplot as plt
from syd import make_viewer

from src.iaf.experiments import get_proximal_experiment
from src.iaf.analysis import (
    build_single_run_metadata,
    gather_num_connections,
    gather_weights,
    gather_results,
    summarize_weights,
    get_groupnames,
)
from src.iaf.optuna_proximal import _normalize_synapses, proximal_weight_entropy, ProximalSearchSpace


# Note:
# -- need to check what's going on with distal synapses and their config -- 
# edge probability, dp_ratio, etc etc
# -- also should validate the distal-pop low entropy results!!!
config = "hofer_all_proximal"
sim, cfg = get_proximal_experiment(
    config,
    num_synapses=3960,
    max_weight=10e-10,
    conductance_threshold=0.3,
    independent_noise_rate=0.15,
    stdp_rate=0.001,
    depression_potentiation_ratio=1.2,
    baseline_rate=2.5,
    driven_rate=90.0,
    concentration=3.0,
    num_simulations=5,
    maintain_distal=True,
    fixed_center=False,
    distal_dp_ratio=1.0,
)

In [6]:
duration = 2400
do_sim = True
if do_sim:
    results = sim.run(duration=duration, initialize=True, save_source_rates=True)

continue_sim = False
if continue_sim:
    continue_duration = 600
    results = sim.run(duration=duration, initialize=False, save_source_rates=True)

In [7]:
# Present for the morning!

In [8]:
average_window = 0.1
entropy = proximal_weight_entropy(results, average_window=average_window)
num_inputs = results["weights"][0]["proximal"].shape[-1]
max_entropy = float(np.log(num_inputs)) if num_inputs > 0 else 1.0
entropy_norm = entropy / max_entropy if max_entropy > 0 else 0.0
synapse_norm = _normalize_synapses(
    cfg.synapses["proximal"].num_synapses,
    ProximalSearchSpace.num_synapses_max,
)


synapse_weight = 0.1
score = entropy_norm * (1 - synapse_weight) + synapse_weight * (1.0 - synapse_norm)

print("Entropy: ", entropy)
print("Entropy norm: ", entropy_norm)
print("Synapse norm: ", synapse_norm)
print("Score: ", score)

Entropy:  2.8048172999787986
Entropy norm:  0.7826991703264725
Synapse norm:  0.9166666666666666
Score:  0.7127625866271586


In [9]:
def make_psth(spike_times, duration, dt):
    psth = np.zeros((duration * int(1/dt)))
    psth[spike_times] = 1
    psth = np.sum(np.reshape(psth, (duration, -1)), axis=1)
    return psth

In [11]:
def plot(state):
    idx_neuron = state["neuron"]
    average_window = state["average_window"]
    weights = results["weights"][idx_neuron]["proximal"]
    num_average = int(average_window * average_window)
    average = np.mean(weights[-num_average:], axis=0)

    spks = results["spike_times"][idx_neuron]
    psth = make_psth(spks, duration, cfg.dt)
    fig, ax = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")
    ax[0].plot(average)
    ax[1].plot(psth)
    return fig

viewer = make_viewer(plot)
viewer.add_integer("neuron", value=0, min=0, max=len(results["weights"]) - 1)
viewer.add_float("average_window", value=0.1, min=0.0, max=1.0)
viewer.show()

In [12]:
from src.iaf.plotting import create_gabor, stitch_gabor_grid

def weights_to_gabor(weights, orientations, spacing=2, **params):
    weights = weights.T.reshape(9, 4)
    gabors = [[None for _ in range(weights.shape[1])] for _ in range(weights.shape[0])]
    for i in range(weights.shape[0]):
        for j in range(weights.shape[1]):
            gabor = weights[i, j] * create_gabor(orientation=orientations[j], **params)
            gabors[i][j] = gabor
        gabors[i] = np.sum(np.stack(gabors[i]), axis=0)
    gwidth = gabors[0].shape[0]
    gabors = np.stack(gabors).reshape(3, 3, gwidth, gwidth)
    return stitch_gabor_grid(gabors, spacing=spacing)

In [13]:
gabor = sim.source_populations["excitatory"]

def plot(state):
    ineuron = state["neuron"]
    spacing = state["spacing"]
    vmax = state["vmax"]

    weights = results["weights"][ineuron]["proximal"]

    average_window = 0.1
    weights = results["weights"][ineuron]["proximal"]
    num_average = int(average_window * average_window)
    average = np.mean(weights[-num_average:], axis=0)

    norm_factor = cfg.synapses["proximal"].max_weight
    num_synapses = cfg.synapses["proximal"].num_synapses
    norm_factor = norm_factor / (36 / num_synapses)
    average = average / norm_factor

    print(np.max(average), num_synapses)

    basal_gabor = weights_to_gabor(average, gabor.orientations, spacing=spacing)
    vmax = np.max(np.abs(basal_gabor))

    fig, ax = plt.subplots(1, 1, figsize=(3, 3), layout="constrained")
    ax.imshow(basal_gabor, vmin=-vmax, vmax=vmax, cmap="bwr")
    ax.set_title(f"{np.max(basal_gabor)}")
    return fig

viewer = make_viewer(plot)
viewer.add_integer("neuron", value=0, min=0, max=len(results["weights"]) - 1)
viewer.add_integer("spacing", value=3, min=0, max=10)
viewer.add_float("vmax", value=0.5, min=0.0, max=1.0)
viewer.show()

0.06889005320333179 3960


In [14]:
sim.neurons[0].synapse_groups["distal-complex"].plasticity_params

PlasticityParams(use_stdp=True, stdp_rate=0.01, depression_potentiation_ratio=1.0, potentiation_tau=0.02, depression_tau=0.02, use_homeostasis=True, homeostasis_tau=20.0, homeostasis_scale=1.0)

In [ ]:
if "replacement" in config:
    norm_by_max_weight = True
    norm_by_num_synapses = False
    norm_by_total_synapses = True
else:
    norm_by_max_weight = True
    norm_by_num_synapses = True
    norm_by_total_synapses = False
        
distal_dp_ratio = 1.1
metadata = build_single_run_metadata(results, sim, config, experiment_type="hofer", dp_ratio=distal_dp_ratio, edge_probability=0.75)
num_connections = gather_num_connections(metadata, experiment_type="hofer")
weights = gather_weights(
    metadata,
    experiment_type="hofer",
    average_method="fraction",
    average_window=0.2,
    norm_by_max_weight=norm_by_max_weight,
    norm_by_num_synapses=norm_by_num_synapses,
    norm_by_total_synapses=norm_by_total_synapses,
    num_connections=num_connections,
)

duration = results["weights"][0]["proximal"].shape[0]
dt = cfg.dt

psths = np.stack([make_psth(st, duration, dt) for st in results["spike_times"]], axis=0)
basal_weights = weights["proximal"][0][0][0]
simple_weights = weights["distal-simple"][0][0][0]
complex_weights = weights["distal-complex"][0][0][0]

gabor = sim.source_populations["excitatory"]

def plot(state):
    ineuron = state["neuron"]
    spacing = state["spacing"]
    vmax = state["vmax"]

    basal_gabor = weights_to_gabor(basal_weights[ineuron], gabor.orientations, spacing=spacing)
    simple_gabor = weights_to_gabor(simple_weights[ineuron], gabor.orientations, spacing=spacing)
    complex_gabor = weights_to_gabor(complex_weights[ineuron], gabor.orientations, spacing=spacing)

    fig, ax = plt.subplots(1, 3, figsize=(9, 3), layout="constrained")
    ax[0].imshow(basal_gabor, vmin=-vmax, vmax=vmax, cmap="bwr")
    ax[0].set_title(f"{np.max(basal_gabor)}")
    ax[1].imshow(simple_gabor, vmin=-vmax, vmax=vmax, cmap="bwr")
    ax[1].set_title(f"{np.max(simple_gabor)}")
    ax[2].imshow(complex_gabor, vmin=-vmax, vmax=vmax, cmap="bwr")
    ax[2].set_title(f"{np.max(complex_gabor)}")
    return fig

viewer = make_viewer(plot)
viewer.add_integer("neuron", value=0, min=0, max=psths.shape[0] - 1)
viewer.add_integer("spacing", value=3, min=0, max=10)
viewer.add_float("vmax", value=0.5, min=0.0, max=1.0)
viewer.show()